# DIFUSCO 论文复现（Jupyter Notebook 完整流程）

本 Notebook 提供一个可直接在 Jupyter 中执行和修改的 DIFUSCO 复现模板流程，覆盖：
1. 环境准备
2. 数据准备
3. 训练（以 TSP100 为例）
4. 评估
5. 结果记录与常见排查

> 命令基于仓库 `README.md` 与 `reproducing_scripts.md`。


In [ ]:
from pathlib import Path
import os
import subprocess
import textwrap

REPO_ROOT = Path.cwd()
print(f"当前工作目录: {REPO_ROOT}")

def run_cmd(cmd: str, check: bool = True):
    print("\n>>>", cmd)
    result = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print(result.stderr)
    if check and result.returncode != 0:
        raise RuntimeError(f"命令执行失败 (code={result.returncode}): {cmd}")
    return result


## 1) 环境准备

```bash
conda env create -f environment.yml
conda activate difusco
```


In [ ]:
run_cmd("python --version", check=False)
run_cmd("python -c 'import torch; print(\"torch\", torch.__version__)'", check=False)
run_cmd("python -c 'import pytorch_lightning as pl; print(\"pytorch_lightning\", pl.__version__)'", check=False)


## 2) 编译 TSP 所需 Cython 模块

根据仓库 README，TSP 实验需要先编译 `difusco/utils/cython_merge`。


In [ ]:
run_cmd("cd difusco/utils/cython_merge && python setup.py build_ext --inplace")


## 3) 路径与实验参数配置（请按需修改）


In [ ]:
STORAGE_PATH = "/your/storage/path"
TRAIN_SPLIT = "/your/tsp100_train_concorde.txt"
VALID_SPLIT = "/your/tsp100_valid_concorde.txt"
TEST_SPLIT = "/your/tsp100_test_concorde.txt"

CUDA_VISIBLE_DEVICES = "0"
WANDB_RUN_ID = "manual_notebook_run"

os.environ["PYTHONPATH"] = f"{REPO_ROOT}:{os.environ.get('PYTHONPATH','')}"
os.environ["CUDA_VISIBLE_DEVICES"] = CUDA_VISIBLE_DEVICES
os.environ["WANDB_RUN_ID"] = WANDB_RUN_ID


## 4) 训练 + 测试（TSP100 复现命令）


In [ ]:
train_cmd = textwrap.dedent(f"""
python -u difusco/train.py \
  --task tsp \
  --wandb_logger_name tsp_diffusion_graph_categorical_tsp100 \
  --diffusion_type categorical \
  --do_train \
  --do_test \
  --learning_rate 0.0002 \
  --weight_decay 0.0001 \
  --lr_scheduler cosine-decay \
  --storage_path {STORAGE_PATH} \
  --training_split {TRAIN_SPLIT} \
  --validation_split {VALID_SPLIT} \
  --test_split {TEST_SPLIT} \
  --batch_size 32 \
  --num_epochs 50 \
  --validation_examples 8 \
  --inference_schedule cosine \
  --inference_diffusion_steps 50
""").strip()
print(train_cmd)
# run_cmd(train_cmd)


## 5) 使用 checkpoint 进行独立评估


In [ ]:
CKPT_PATH = "/your/tsp100_categorical/ckpt_path/last.ckpt"

eval_cmd = textwrap.dedent(f"""
python -u difusco/train.py \
  --task tsp \
  --wandb_logger_name tsp_diffusion_graph_categorical_tsp100_test \
  --diffusion_type categorical \
  --do_test \
  --learning_rate 0.0002 \
  --weight_decay 0.0001 \
  --lr_scheduler cosine-decay \
  --storage_path {STORAGE_PATH} \
  --training_split {TRAIN_SPLIT} \
  --validation_split {VALID_SPLIT} \
  --test_split {TEST_SPLIT} \
  --batch_size 32 \
  --num_epochs 25 \
  --inference_schedule cosine \
  --inference_diffusion_steps 50 \
  --ckpt_path {CKPT_PATH} \
  --resume_weight_only
""").strip()
print(eval_cmd)
# run_cmd(eval_cmd)


## 6) 常见问题排查

1. 找不到数据文件：检查路径参数。
2. 显存不足：减小 `batch_size`。
3. TSP 推理报错：确认已编译 `cython_merge`。
